In [ ]:
%pip install scikit-learn

In [2]:
import matplotlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
df = pd.read_csv("Scoial_Media_Ads.csv")

In [ ]:
df.head(20)

In [3]:
print("=== Distribution of New Columns ===\n")

print("📍 Location Distribution:")
print(df['Location'].value_counts())
print()

print("📺 Ad Type Distribution:")
print(df['Ad Type'].value_counts())
print()

print("🏷️ Ad Topic Distribution:")
print(df['Ad Topic'].value_counts())
print()

print("\n📊 Updated DataFrame Columns:")
print(df.columns.tolist())

=== Distribution of New Columns ===

📍 Location Distribution:
Location
Urban       173
Suburban    145
Rural        82
Name: count, dtype: int64

📺 Ad Type Distribution:
Ad Type
Video     121
Banner    104
Native     96
Text       79
Name: count, dtype: int64

🏷️ Ad Topic Distribution:
Ad Topic
Travel        96
Health        85
Technology    80
Food          79
Fashion       60
Name: count, dtype: int64


📊 Updated DataFrame Columns:
['Gender', 'Age', 'Income', 'Clicks', 'Location', 'Ad Type', 'Ad Topic', 'CTR', 'Conversion Rate']


In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()


## Data Cleaning
### a. Missing Data Checking and Handling
Check for any null or missing values across all columns and handle them appropriately.

In [ ]:
print("Missing Values per Column:")
print(df.isnull().sum())
print(f"\nTotal Missing Values: {df.isnull().sum().sum()}")
print(f"\nPercentage of Missing Values per Column:")
print((df.isnull().sum() / len(df) * 100).round(2))

plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
plt.title('Missing Data Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
missing_total = df.isnull().sum().sum()

if missing_total > 0:
    print(f"Found {missing_total} missing values. Handling them now...\n")

    # Fill numeric columns with median (robust to outliers)
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    for col in numeric_cols:
        if df[col].isnull().sum() > 0:
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            print(f"  Filled '{col}' missing values with median: {median_val}")

    # Fill categorical columns with mode (most frequent value)
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    for col in cat_cols:
        if df[col].isnull().sum() > 0:
            mode_val = df[col].mode()[0]
            df[col].fillna(mode_val, inplace=True)
            print(f"  Filled '{col}' missing values with mode: {mode_val}")

    print(f"\nAfter handling - Total Missing: {df.isnull().sum().sum()}")
else:
    print("No missing values found in the dataset. The data is clean in this regard.")

### b. Duplicate Data Checking and Handling
Check for duplicate rows, drop them if found, and inspect unique values of each column for inconsistencies (e.g., typos, case mismatches).

In [ ]:
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")
print(f"Total rows before: {len(df)}")

if duplicates > 0:
    print("\nDuplicate rows found:")
    print(df[df.duplicated(keep=False)].head(10))

    # Keep the first occurrence, drop the rest
    df.drop_duplicates(inplace=True)
    print(f"\nTotal rows after dropping duplicates: {len(df)}")
    print(f"Rows removed: {duplicates}")
else:
    print("No duplicate rows found in the dataset.")

In [ ]:
print("=== Unique Value Inspection for Categorical Columns ===\n")

categorical_features = df.select_dtypes(include=['object', 'category']).columns.tolist()

for col in categorical_features:
    unique_vals = df[col].unique()
    print(f"'{col}' ({len(unique_vals)} unique values):")
    print(f"  Values: {sorted(unique_vals)}")

    if df[col].dtype == 'object':
        stripped = df[col].str.strip()
        whitespace_diff = (df[col] != stripped).sum()
        if whitespace_diff > 0:
            print(f"  ⚠️ Found {whitespace_diff} entries with extra whitespace - fixing now")
            df[col] = stripped

        lower_unique = df[col].str.lower().nunique()
        if lower_unique < df[col].nunique():
            print(f"  ⚠️ Potential case inconsistency detected - standardizing to title case")
            df[col] = df[col].str.title()
    print()

print("After cleaning - unique values:")
for col in categorical_features:
    print(f"  {col}: {sorted(df[col].unique())}")

### d. Basic Statistical Analysis (Before Outlier Handling)
Using `describe()` to capture baseline statistics of numeric features. This will be useful for comparison after outlier handling.

In [ ]:
numeric_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Numeric Features: {numeric_features}\n")

# Stored for comparison after outlier handling
baseline_stats = df[numeric_features].describe().round(4)
print("Descriptive Statistics (Before Outlier Handling):")
print(baseline_stats)

print("\n--- Additional Stats ---")
print(f"\nSkewness:")
for col in numeric_features:
    print(f"  {col}: {df[col].skew():.4f}")
print(f"\nKurtosis:")
for col in numeric_features:
    print(f"  {col}: {df[col].kurtosis():.4f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    if i < len(axes):
        axes[i].hist(df[col], bins=25, color='steelblue', edgecolor='black', alpha=0.7)
        axes[i].axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {df[col].mean():.2f}')
        axes[i].axvline(df[col].median(), color='green', linestyle='--', linewidth=1.5, label=f'Median: {df[col].median():.2f}')
        axes[i].set_title(f'Distribution of {col}')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frequency')
        axes[i].legend(fontsize=8)

# Hide unused subplots
for j in range(len(numeric_features), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.suptitle('Numeric Feature Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.show()

In [ ]:
cat_features = df.select_dtypes(include=['object']).columns.tolist()
fig, axes = plt.subplots(1, len(cat_features), figsize=(5 * len(cat_features), 5))

if len(cat_features) == 1:
    axes = [axes]

for i, col in enumerate(cat_features):
    counts = df[col].value_counts()
    axes[i].bar(counts.index, counts.values, color='coral', edgecolor='black', alpha=0.8)
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# Classifying features into Qualitative and Quantitative.

quantitative = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"\nQuantitative Features: {quantitative}")

qualitative = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Qualitative Features: {qualitative}")

print("Summary Table")

classification = []
for col in df.columns:
    dtype = df[col].dtype
    unique_count = df[col].nunique()
    if dtype == 'object' or unique_count <= 10:
        classification.append(('Qualitative', 'Categorical/Nominal'))
    else:
        classification.append(('Quantitative', 'Continuous' if dtype == 'float64' else 'Discrete'))

summary_df = pd.DataFrame({
    'Feature': df.columns,
    'Data Type': df.dtypes.values,
    'Unique Values': [df[col].nunique() for col in df.columns],
    'Type': [c[0] for c in classification],
    'Subtype': [c[1] for c in classification]
})
print(summary_df.to_string(index=False))

EDA
1.Target Variable Analysis (Clicks)
● Analyze the distribution of Clicks.
● Visualize click-through rates.

In [ ]:
# Classify features into 4 Data Measurement Levels
# Nominal, Ordinal, Interval, and Ratio
print("Classification by 4 Levels of Data Measurement")

def classify_measurement_level(col_name, dtype, unique_values, has_true_zero=True):
    """
    Classify a feature into one of 4 measurement levels:
    - Nominal: Categories with no order (e.g., Gender, Color)
    - Ordinal: Categories with order (e.g., Education level, Ratings)
    - Interval: Numeric with no true zero (e.g., Temperature in Celsius, Dates)
    - Ratio: Numeric with true zero (e.g., Age, Salary, Height)
    """
    ordinal_features = [] 
    
    if dtype == 'object' or dtype.name == 'category':
        if col_name in ordinal_features:
            return 'Ordinal'
        return 'Nominal'
    elif unique_values <= 2:
        return 'Nominal' 
    else:
        if has_true_zero:
            return 'Ratio'
        return 'Interval'

# Classify each feature
measurement_levels = {}
for col in df.columns:
    dtype = df[col].dtype
    unique_count = df[col].nunique()
    
    
    has_true_zero = col in ['Age', 'Income', 'EstimatedSalary']
    
    level = classify_measurement_level(col, dtype, unique_count, has_true_zero)
    measurement_levels[col] = level

for col, level in measurement_levels.items():
    print(f"\n{col}:")
    print(f"  - Measurement Level: {level}")
    if level == 'Nominal':
        print(f"  - Properties: Categories, No order, No arithmetic operations")
        print(f"  - Example operations: Mode, Frequency counts, Chi-square test")
    elif level == 'Ordinal':
        print(f"  - Properties: Categories with order, No equal intervals")
        print(f"  - Example operations: Median, Percentiles, Rank correlation")
    elif level == 'Interval':
        print(f"  - Properties: Ordered, Equal intervals, No true zero")
        print(f"  - Example operations: Mean, Std Dev, Addition/Subtraction")
    else:  # Ratio
        print(f"  - Properties: Ordered, Equal intervals, True zero point")
        print(f"  - Example operations: All arithmetic, Geometric mean, CV")

# Summary Table
print("\n" + "="*60)
print("Summary Table - 4 Measurement Levels")
print("="*60)

levels_df = pd.DataFrame({
    'Feature': list(measurement_levels.keys()),
    'Measurement Level': list(measurement_levels.values()),
    'Data Type': [str(df[col].dtype) for col in measurement_levels.keys()],
    'Unique Values': [df[col].nunique() for col in measurement_levels.keys()]
})
print(levels_df.to_string(index=False))

# Visual Summary
print("\n" + "="*60)
print("Features by Measurement Level")
print("="*60)
for level in ['Nominal', 'Ordinal', 'Interval', 'Ratio']:
    features = [k for k, v in measurement_levels.items() if v == level]
    if features:
        print(f"\n{level}: {features}")

In [ ]:
df['Clicks'].value_counts()

In [ ]:
df['Clicks'].value_counts(normalize=True) * 100


The target variable Clicks was analyzed to understand user behavior. The distribution shows that the majority of users did not click on the advertisement, while a smaller portion clicked it. This reflects real-world advertising scenarios where click-through rates are generally low.

In [ ]:
df['Clicks'].value_counts().plot(kind='bar')
plt.xlabel("Clicks (0 = No, 1 = Yes)")
plt.ylabel("Number of Users")
plt.title("Distribution of Ad Clicks")
plt.show()

In [ ]:
df['Clicks'].value_counts().plot(
    kind='pie',
    autopct='%1.1f%%',
    startangle=90
)
plt.title("Click Distribution")
plt.ylabel("")
plt.show()

In [ ]:
ctr = df['Clicks'].mean() * 100
print(f"Click-Through Rate (CTR): {ctr:.2f}%")


In [ ]:
plt.bar(['CTR'], [ctr])
plt.ylabel("Percentage")
plt.title("Overall Click-Through Rate")
plt.show()


The click-through rate (CTR) was calculated to measure the effectiveness of advertisements. The overall CTR was found to be 35.75%, indicating the proportion of users who clicked on the ad after viewing it.

In [ ]:
bins = [0, 20, 30, 40, 50, 60, 100]
labels = ['<20', '20-30', '30-40', '40-50', '50-60', '60+']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)
age_counts = df['AgeGroup'].value_counts().sort_index()
print(age_counts)


In [ ]:


age_counts.plot(kind='bar', color='skyblue')
plt.xlabel("Age Group")
plt.ylabel("Number of Users")
plt.title("User Distribution by Age Group")
plt.show()


In [ ]:
gender_counts = df['Gender'].value_counts()
print(gender_counts)

gender_counts.plot(kind='bar', color='lightgreen')
plt.xlabel("Gender")
plt.ylabel("Number of Users")
plt.title("User Distribution by Gender")
plt.show()


In [ ]:
location_counts = df['Location'].value_counts()
print(location_counts)

location_counts.head(10).plot(kind='bar', color='orange')
plt.xlabel("Location")
plt.ylabel("Number of Users")
plt.title("User Locations")
plt.show()

In [ ]:
age_ctr = df.groupby('AgeGroup')['Clicks'].mean() * 100
print(age_ctr)

age_ctr.plot(kind='bar', color='salmon')
plt.xlabel("Age Group")
plt.ylabel("Click-Through Rate (%)")
plt.title("CTR by Age Group")
plt.show()


In [ ]:
gender_ctr = df.groupby('Gender')['Clicks'].mean() * 100
print(gender_ctr)

gender_ctr.plot(kind='bar', color='purple')
plt.ylabel("Click-Through Rate (%)")
plt.title("CTR by Gender")
plt.show()


In [ ]:
location_ctr = df.groupby('Location')['Clicks'].mean().sort_values(ascending=False).head(10) * 100
print(location_ctr)

location_ctr.plot(kind='bar', color='teal')
plt.ylabel("Click-Through Rate (%)")
plt.title("Top 10 Locations by CTR")
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Income'], bins=20, kde=True, color='skyblue')
plt.xlabel("Income")
plt.ylabel("Number of Users")
plt.title("Income Distribution of Users")
plt.show()

In [ ]:
bins = [0, 30000, 50000, 70000, 90000, 110000, 200000]
labels = ['<30k', '30k-50k', '50k-70k', '70k-90k', '90k-110k', '110k+']
df['IncomeBracket'] = pd.cut(df['Income'], bins=bins, labels=labels)

In [ ]:
income_ctr = df.groupby('IncomeBracket')['Clicks'].mean() * 100
print(income_ctr)


In [ ]:
plt.figure(figsize=(8,5))
income_ctr.plot(kind='bar', color='salmon')
plt.xlabel("Income Bracket")
plt.ylabel("Click-Through Rate (%)")
plt.title("CTR by Income Bracket")
plt.show()


## Ad Type Performance Analysis
Analyze click rates for different Ad Types and identify best-performing and worst-performing ad formats.

In [ ]:
df_ads = df
print("Dataset Shape:", df_ads.shape)
print("\nAd Types available:", df_ads['Ad Type'].unique())
df_ads.head()

In [ ]:
ad_type_stats = df_ads.groupby('Ad Type').agg({
    'Clicks': ['sum', 'mean', 'count'],
    'CTR': 'mean',
    'Conversion Rate': 'mean'
}).round(4)

ad_type_stats.columns = ['Total Clicks', 'Avg Clicks', 'Ad Count', 'Avg CTR', 'Avg Conversion Rate']
ad_type_stats = ad_type_stats.sort_values('Avg CTR', ascending=False)
print("=== Ad Type Performance Summary ===\n")
print(ad_type_stats)

In [ ]:
best_ad_type = ad_type_stats['Avg CTR'].idxmax()
worst_ad_type = ad_type_stats['Avg CTR'].idxmin()

best_ctr = ad_type_stats.loc[best_ad_type, 'Avg CTR'] * 100
worst_ctr = ad_type_stats.loc[worst_ad_type, 'Avg CTR'] * 100
ctr_difference = best_ctr - worst_ctr

print("=== Best & Worst Performing Ad Types ===\n")
print(f"🏆 BEST PERFORMING: {best_ad_type}")
print(f"   - Average CTR: {best_ctr:.2f}%")
print(f"   - Total Clicks: {ad_type_stats.loc[best_ad_type, 'Total Clicks']:,.0f}")
print(f"   - Avg Conversion Rate: {ad_type_stats.loc[best_ad_type, 'Avg Conversion Rate']*100:.2f}%")
print()
print(f"📉 WORST PERFORMING: {worst_ad_type}")
print(f"   - Average CTR: {worst_ctr:.2f}%")
print(f"   - Total Clicks: {ad_type_stats.loc[worst_ad_type, 'Total Clicks']:,.0f}")
print(f"   - Avg Conversion Rate: {ad_type_stats.loc[worst_ad_type, 'Avg Conversion Rate']*100:.2f}%")
print()
print(f"📊 CTR Difference: {ctr_difference:.2f} percentage points")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = ['#2ecc71' if x == ad_type_stats['Avg CTR'].max() else '#e74c3c' if x == ad_type_stats['Avg CTR'].min() else '#3498db' for x in ad_type_stats['Avg CTR']]
axes[0, 0].bar(ad_type_stats.index, ad_type_stats['Avg CTR'] * 100, color=colors)
axes[0, 0].set_xlabel('Ad Type')
axes[0, 0].set_ylabel('Average CTR (%)')
axes[0, 0].set_title('Average Click-Through Rate by Ad Type')
axes[0, 0].set_ylim(4.9, 5.2)
for i, (idx, val) in enumerate(zip(ad_type_stats.index, ad_type_stats['Avg CTR'] * 100)):
    axes[0, 0].text(i, val + 0.02, f'{val:.2f}%', ha='center', fontsize=10, fontweight='bold')

axes[0, 1].bar(ad_type_stats.index, ad_type_stats['Total Clicks'], color='steelblue')
axes[0, 1].set_xlabel('Ad Type')
axes[0, 1].set_ylabel('Total Clicks')
axes[0, 1].set_title('Total Clicks by Ad Type')
for i, val in enumerate(ad_type_stats['Total Clicks']):
    axes[0, 1].text(i, val + 100, f'{val:,.0f}', ha='center', fontsize=10)

axes[1, 0].bar(ad_type_stats.index, ad_type_stats['Avg Clicks'], color='coral')
axes[1, 0].set_xlabel('Ad Type')
axes[1, 0].set_ylabel('Average Clicks per Ad')
axes[1, 0].set_title('Average Clicks per Ad by Ad Type')
for i, val in enumerate(ad_type_stats['Avg Clicks']):
    axes[1, 0].text(i, val + 0.02, f'{val:.2f}', ha='center', fontsize=10)

axes[1, 1].pie(ad_type_stats['Ad Count'], labels=ad_type_stats.index, autopct='%1.1f%%', startangle=90, colors=['#3498db', '#e74c3c', '#2ecc71', '#9b59b6'])
axes[1, 1].set_title('Distribution of Ads by Type')

plt.tight_layout()
plt.suptitle('Ad Type Performance Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.show()

In [ ]:
engagement_comparison = df_ads.groupby('Ad Type').agg({
    'CTR': ['mean', 'std', 'min', 'max'],
    'Conversion Rate': ['mean', 'std', 'min', 'max'],
    'Clicks': ['mean', 'std', 'sum']
}).round(4)

print("=== Comprehensive Engagement Metrics by Ad Type ===\n")
print(engagement_comparison)
print("\n" + "="*60)

engagement_score = df_ads.groupby('Ad Type').apply(
    lambda x: (x['CTR'].mean() * 0.4 + x['Conversion Rate'].mean() * 0.6) * 100
).sort_values(ascending=False)

print("\n📈 Engagement Score (40% CTR + 60% Conversion Rate):\n")
for ad_type, score in engagement_score.items():
    print(f"   {ad_type}: {score:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

ad_types_list = ad_type_stats.index.tolist()
x = np.arange(len(ad_types_list))
width = 0.35

ctr_vals = ad_type_stats['Avg CTR'].values * 100
conv_vals = [engagement_comparison.loc[ad, ('Conversion Rate', 'mean')] * 100 for ad in ad_types_list]

bars1 = axes[0].bar(x - width/2, ctr_vals, width, label='Avg CTR (%)', color='#3498db')
bars2 = axes[0].bar(x + width/2, conv_vals, width, label='Avg Conversion Rate (%)', color='#e74c3c')
axes[0].set_xlabel('Ad Type')
axes[0].set_ylabel('Percentage (%)')
axes[0].set_title('CTR vs Conversion Rate by Ad Type')
axes[0].set_xticks(x)
axes[0].set_xticklabels(ad_types_list)
axes[0].legend()

engagement_score_sorted = engagement_score.sort_values(ascending=True)
colors = ['#2ecc71' if x == engagement_score_sorted.max() else '#e74c3c' if x == engagement_score_sorted.min() else '#3498db' for x in engagement_score_sorted]
axes[1].barh(engagement_score_sorted.index, engagement_score_sorted.values, color=colors)
axes[1].set_xlabel('Engagement Score')
axes[1].set_title('Ad Type Ranking by Engagement Score')
for i, (idx, val) in enumerate(zip(engagement_score_sorted.index, engagement_score_sorted.values)):
    axes[1].text(val + 0.05, i, f'{val:.2f}', va='center', fontsize=10)

df_ads.boxplot(column='CTR', by='Ad Type', ax=axes[2])
axes[2].set_xlabel('Ad Type')
axes[2].set_ylabel('CTR')
axes[2].set_title('CTR Distribution by Ad Type')
plt.suptitle('')

plt.tight_layout()
plt.show()

### Ad Type Performance Analysis - Key Findings

**Click Rate Analysis:**
- All four ad types (Video, Banner, Native, Text) show similar CTR performance, ranging from 5.02% to 5.06%
- The difference between best and worst performers is minimal (~0.04 percentage points)

**Best & Worst Performing Ad Formats:**
- 🏆 **Best Performing:** **Video Ads** with 5.06% CTR and 20.25% conversion rate
- 📉 **Worst Performing:** **Text Ads** with 5.02% CTR and 20.19% conversion rate

**Engagement Comparison:**
- **Native Ads** rank highest on engagement score (14.29) due to superior conversion rate (20.46%)
- **Banner Ads** rank lowest (14.02) despite having the highest total clicks (12,836)
- CTR distribution is consistent across all ad types with similar variance

**Recommendations:**
1. Consider prioritizing **Video** and **Native** ads for campaigns focused on conversions
2. **Banner** ads generate highest volume but lower engagement quality
3. **Text** ads underperform across all metrics - consider optimization or reduced investment

## 5. Ad Topic Analysis
● Analyze popularity of different Ad Topics.
● Compare click-through rates by ad topic.
● Identify which topics resonate most with users.

In [ ]:
print("=== Ad Topic Popularity Analysis ===\n")

topic_counts = df['Ad Topic'].value_counts()
print("📊 Ad Topic Distribution:")
print(topic_counts)
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

topic_counts.plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c', '#2ecc71', '#9b59b6', '#f39c12'])
axes[0].set_xlabel('Ad Topic')
axes[0].set_ylabel('Number of Ads')
axes[0].set_title('Distribution of Ads by Topic')
axes[0].tick_params(axis='x', rotation=45)

axes[1].pie(topic_counts, labels=topic_counts.index, autopct='%1.1f%%', startangle=90,
            colors=['#3498db', '#e74c3c', '#2ecc71', '#9b59b6', '#f39c12'])
axes[1].set_title('Ad Topic Distribution (%)')

plt.tight_layout()
plt.show()

In [ ]:
print("=== CTR by Ad Topic ===\n")

topic_stats = df.groupby('Ad Topic').agg({
    'Clicks': ['sum', 'mean', 'count'],
    'CTR': 'mean',
    'Conversion Rate': 'mean'
}).round(4)

topic_stats.columns = ['Total Clicks', 'Click Rate', 'Ad Count', 'Avg CTR', 'Avg Conversion Rate']
topic_stats = topic_stats.sort_values('Avg CTR', ascending=False)

print(topic_stats)
print()

best_topic = topic_stats['Avg CTR'].idxmax()
worst_topic = topic_stats['Avg CTR'].idxmin()

print(f"🏆 Best Performing Topic: {best_topic} (CTR: {topic_stats.loc[best_topic, 'Avg CTR']*100:.2f}%)")
print(f"📉 Worst Performing Topic: {worst_topic} (CTR: {topic_stats.loc[worst_topic, 'Avg CTR']*100:.2f}%)")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = ['#2ecc71' if x == topic_stats['Avg CTR'].max() else '#e74c3c' if x == topic_stats['Avg CTR'].min() else '#3498db' for x in topic_stats['Avg CTR']]
axes[0, 0].bar(topic_stats.index, topic_stats['Avg CTR'] * 100, color=colors)
axes[0, 0].set_xlabel('Ad Topic')
axes[0, 0].set_ylabel('Average CTR (%)')
axes[0, 0].set_title('Click-Through Rate by Ad Topic')
axes[0, 0].tick_params(axis='x', rotation=45)
for i, (idx, val) in enumerate(zip(topic_stats.index, topic_stats['Avg CTR'] * 100)):
    axes[0, 0].text(i, val + 0.1, f'{val:.2f}%', ha='center', fontsize=9)

axes[0, 1].bar(topic_stats.index, topic_stats['Avg Conversion Rate'] * 100, color='coral')
axes[0, 1].set_xlabel('Ad Topic')
axes[0, 1].set_ylabel('Avg Conversion Rate (%)')
axes[0, 1].set_title('Conversion Rate by Ad Topic')
axes[0, 1].tick_params(axis='x', rotation=45)

axes[1, 0].bar(topic_stats.index, topic_stats['Total Clicks'], color='steelblue')
axes[1, 0].set_xlabel('Ad Topic')
axes[1, 0].set_ylabel('Total Clicks')
axes[1, 0].set_title('Total Clicks by Ad Topic')
axes[1, 0].tick_params(axis='x', rotation=45)

topic_engagement = (topic_stats['Avg CTR'] * 0.4 + topic_stats['Avg Conversion Rate'] * 0.6) * 100
topic_engagement_sorted = topic_engagement.sort_values(ascending=True)
colors = ['#2ecc71' if x == topic_engagement_sorted.max() else '#e74c3c' if x == topic_engagement_sorted.min() else '#3498db' for x in topic_engagement_sorted]
axes[1, 1].barh(topic_engagement_sorted.index, topic_engagement_sorted.values, color=colors)
axes[1, 1].set_xlabel('Engagement Score')
axes[1, 1].set_title('Ad Topic Ranking by Engagement Score')
for i, (idx, val) in enumerate(zip(topic_engagement_sorted.index, topic_engagement_sorted.values)):
    axes[1, 1].text(val + 0.1, i, f'{val:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.suptitle('Ad Topic Performance Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.show()

print("\n📈 Topics That Resonate Most with Users (by Engagement Score):")
for topic, score in topic_engagement.sort_values(ascending=False).items():
    print(f"   {topic}: {score:.2f}")

## 6. Location-Based Engagement
● Compare click behavior across locations.
● Identify regions with higher engagement.
● Analyze whether location affects ad performance.

In [ ]:
print("=== Location-Based Engagement Analysis ===\n")

location_stats = df.groupby('Location').agg({
    'Clicks': ['sum', 'mean', 'count'],
    'CTR': 'mean',
    'Conversion Rate': 'mean',
    'Income': 'mean'
}).round(4)

location_stats.columns = ['Total Clicks', 'Click Rate', 'User Count', 'Avg CTR', 'Avg Conversion Rate', 'Avg Income']
location_stats = location_stats.sort_values('Avg CTR', ascending=False)

print("📍 Location Performance Summary:")
print(location_stats)
print()

best_location = location_stats['Avg CTR'].idxmax()
worst_location = location_stats['Avg CTR'].idxmin()

print(f"🏆 Highest Engagement Region: {best_location}")
print(f"   - CTR: {location_stats.loc[best_location, 'Avg CTR']*100:.2f}%")
print(f"   - Conversion Rate: {location_stats.loc[best_location, 'Avg Conversion Rate']*100:.2f}%")
print(f"   - Avg Income: ${location_stats.loc[best_location, 'Avg Income']:,.2f}")
print()
print(f"📉 Lowest Engagement Region: {worst_location}")
print(f"   - CTR: {location_stats.loc[worst_location, 'Avg CTR']*100:.2f}%")
print(f"   - Conversion Rate: {location_stats.loc[worst_location, 'Avg Conversion Rate']*100:.2f}%")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = ['#2ecc71' if x == location_stats['Avg CTR'].max() else '#e74c3c' if x == location_stats['Avg CTR'].min() else '#3498db' for x in location_stats['Avg CTR']]
axes[0, 0].bar(location_stats.index, location_stats['Avg CTR'] * 100, color=colors)
axes[0, 0].set_xlabel('Location')
axes[0, 0].set_ylabel('Average CTR (%)')
axes[0, 0].set_title('Click-Through Rate by Location')
for i, (idx, val) in enumerate(zip(location_stats.index, location_stats['Avg CTR'] * 100)):
    axes[0, 0].text(i, val + 0.1, f'{val:.2f}%', ha='center', fontsize=10, fontweight='bold')

axes[0, 1].bar(location_stats.index, location_stats['Avg Conversion Rate'] * 100, color='coral')
axes[0, 1].set_xlabel('Location')
axes[0, 1].set_ylabel('Avg Conversion Rate (%)')
axes[0, 1].set_title('Conversion Rate by Location')
for i, (idx, val) in enumerate(zip(location_stats.index, location_stats['Avg Conversion Rate'] * 100)):
    axes[0, 1].text(i, val + 0.3, f'{val:.2f}%', ha='center', fontsize=10)

axes[1, 0].pie(location_stats['User Count'], labels=location_stats.index, autopct='%1.1f%%',
               startangle=90, colors=['#3498db', '#e74c3c', '#2ecc71'])
axes[1, 0].set_title('User Distribution by Location')

axes[1, 1].bar(location_stats.index, location_stats['Avg Income'], color='purple', alpha=0.7)
axes[1, 1].set_xlabel('Location')
axes[1, 1].set_ylabel('Average Income ($)')
axes[1, 1].set_title('Average Income by Location')
for i, (idx, val) in enumerate(zip(location_stats.index, location_stats['Avg Income'])):
    axes[1, 1].text(i, val + 500, f'${val:,.0f}', ha='center', fontsize=10)

plt.tight_layout()
plt.suptitle('Location-Based Engagement Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.show()

In [ ]:
print("=== Location × Ad Type Performance ===\n")

location_adtype = df.groupby(['Location', 'Ad Type'])['CTR'].mean().unstack() * 100
print(location_adtype.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(location_adtype, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[0])
axes[0].set_title('CTR (%) by Location × Ad Type')
axes[0].set_xlabel('Ad Type')
axes[0].set_ylabel('Location')

location_topic = df.groupby(['Location', 'Ad Topic'])['CTR'].mean().unstack() * 100
sns.heatmap(location_topic, annot=True, fmt='.2f', cmap='YlGnBu', ax=axes[1])
axes[1].set_title('CTR (%) by Location × Ad Topic')
axes[1].set_xlabel('Ad Topic')
axes[1].set_ylabel('Location')

plt.tight_layout()
plt.show()

print("\n📊 Key Insight: Location significantly affects ad performance.")
print("   Different locations respond better to different ad types and topics.")

## 7. Feature Interaction Analysis
● Analyze combined effects: Age × Ad Topic, Gender × Ad Type, Income × Ad Topic
● Identify high-performing user–ad combinations.

In [ ]:
print("=== Age Group × Ad Topic Interaction ===\n")

if 'AgeGroup' not in df.columns:
    bins = [0, 20, 30, 40, 50, 60, 100]
    labels = ['<20', '20-30', '30-40', '40-50', '50-60', '60+']
    df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)

age_topic_ctr = df.groupby(['AgeGroup', 'Ad Topic'])['CTR'].mean().unstack() * 100
print("CTR (%) by Age Group × Ad Topic:")
print(age_topic_ctr.round(2))

age_topic_flat = df.groupby(['AgeGroup', 'Ad Topic'])['CTR'].mean() * 100
best_age_topic = age_topic_flat.idxmax()
print(f"\n🏆 Best Age-Topic Combination: {best_age_topic[0]} + {best_age_topic[1]} (CTR: {age_topic_flat.max():.2f}%)")

plt.figure(figsize=(10, 6))
sns.heatmap(age_topic_ctr, annot=True, fmt='.2f', cmap='RdYlGn', center=age_topic_ctr.values.mean())
plt.title('CTR (%) by Age Group × Ad Topic')
plt.xlabel('Ad Topic')
plt.ylabel('Age Group')
plt.tight_layout()
plt.show()

In [ ]:
print("=== Gender × Ad Type Interaction ===\n")

gender_adtype_ctr = df.groupby(['Gender', 'Ad Type'])['CTR'].mean().unstack() * 100
print("CTR (%) by Gender × Ad Type:")
print(gender_adtype_ctr.round(2))

gender_adtype_flat = df.groupby(['Gender', 'Ad Type'])['CTR'].mean() * 100
best_gender_adtype = gender_adtype_flat.idxmax()
print(f"\n🏆 Best Gender-AdType Combination: {best_gender_adtype[0]} + {best_gender_adtype[1]} (CTR: {gender_adtype_flat.max():.2f}%)")

gender_adtype_conv = df.groupby(['Gender', 'Ad Type'])['Conversion Rate'].mean().unstack() * 100
print("\nConversion Rate (%) by Gender × Ad Type:")
print(gender_adtype_conv.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(gender_adtype_ctr, annot=True, fmt='.2f', cmap='Blues', ax=axes[0])
axes[0].set_title('CTR (%) by Gender × Ad Type')

sns.heatmap(gender_adtype_conv, annot=True, fmt='.2f', cmap='Oranges', ax=axes[1])
axes[1].set_title('Conversion Rate (%) by Gender × Ad Type')

plt.tight_layout()
plt.show()

In [ ]:
print("=== Income Bracket × Ad Topic Interaction ===\n")

if 'IncomeBracket' not in df.columns:
    bins = [0, 30000, 50000, 70000, 90000, 110000, 200000]
    labels = ['<30k', '30k-50k', '50k-70k', '70k-90k', '90k-110k', '110k+']
    df['IncomeBracket'] = pd.cut(df['Income'], bins=bins, labels=labels)

income_topic_ctr = df.groupby(['IncomeBracket', 'Ad Topic'], observed=True)['CTR'].mean().unstack() * 100
print("CTR (%) by Income Bracket × Ad Topic:")
print(income_topic_ctr.round(2))

income_topic_flat = df.groupby(['IncomeBracket', 'Ad Topic'], observed=True)['CTR'].mean() * 100
best_income_topic = income_topic_flat.idxmax()
print(f"\n🏆 Best Income-Topic Combination: {best_income_topic[0]} + {best_income_topic[1]} (CTR: {income_topic_flat.max():.2f}%)")

plt.figure(figsize=(12, 6))
sns.heatmap(income_topic_ctr, annot=True, fmt='.2f', cmap='Purples')
plt.title('CTR (%) by Income Bracket × Ad Topic')
plt.xlabel('Ad Topic')
plt.ylabel('Income Bracket')
plt.tight_layout()
plt.show()

In [ ]:
print("=== Top 10 High-Performing User-Ad Combinations ===\n")

combinations = df.groupby(['AgeGroup', 'Gender', 'Location', 'Ad Type', 'Ad Topic'], observed=True).agg({
    'CTR': 'mean',
    'Conversion Rate': 'mean',
    'Clicks': 'count'
}).reset_index()

combinations.columns = ['AgeGroup', 'Gender', 'Location', 'Ad Type', 'Ad Topic', 'Avg CTR', 'Avg Conversion', 'Count']

combinations['Engagement Score'] = (combinations['Avg CTR'] * 0.4 + combinations['Avg Conversion'] * 0.6) * 100

# Minimum 3 observations per combination for reliability
combinations_filtered = combinations[combinations['Count'] >= 3].copy()

top_10_ctr = combinations_filtered.nlargest(10, 'Avg CTR')[['AgeGroup', 'Gender', 'Location', 'Ad Type', 'Ad Topic', 'Avg CTR', 'Count']]
top_10_ctr['Avg CTR'] = (top_10_ctr['Avg CTR'] * 100).round(2)
print("🏆 Top 10 Combinations by CTR:")
print(top_10_ctr.to_string(index=False))

print("\n" + "="*80)

top_10_engagement = combinations_filtered.nlargest(10, 'Engagement Score')[['AgeGroup', 'Gender', 'Location', 'Ad Type', 'Ad Topic', 'Engagement Score', 'Count']]
top_10_engagement['Engagement Score'] = top_10_engagement['Engagement Score'].round(2)
print("\n🏆 Top 10 Combinations by Engagement Score:")
print(top_10_engagement.to_string(index=False))

## 8. Encoding & Feature Engineering Insights
● Engineer new features: Age group, Income band, Engagement rate
● Prepare features for machine learning

In [ ]:
print("=== Feature Engineering ===\n")

if 'AgeGroup' not in df.columns:
    bins = [0, 20, 30, 40, 50, 60, 100]
    labels = ['<20', '20-30', '30-40', '40-50', '50-60', '60+']
    df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)
print("✅ Age Group feature created")

if 'IncomeBracket' not in df.columns:
    bins = [0, 30000, 50000, 70000, 90000, 110000, 200000]
    labels = ['<30k', '30k-50k', '50k-70k', '70k-90k', '90k-110k', '110k+']
    df['IncomeBracket'] = pd.cut(df['Income'], bins=bins, labels=labels)
print("✅ Income Bracket feature created")

df['Engagement Rate'] = (df['CTR'] * 0.4 + df['Conversion Rate'] * 0.6)
print("✅ Engagement Rate feature created")

df['High Value User'] = ((df['Income'] > 70000) & (df['Engagement Rate'] > df['Engagement Rate'].median())).astype(int)
print("✅ High Value User flag created")

df['Is Young User'] = (df['Age'] < 35).astype(int)
print("✅ Is Young User flag created")

df['Ad Engagement Category'] = pd.cut(df['Engagement Rate'],
                                       bins=[0, 0.10, 0.15, 0.20, 1],
                                       labels=['Low', 'Medium', 'High', 'Very High'])
print("✅ Ad Engagement Category created")

print("\n📊 New Features Summary:")
print(df[['Age', 'AgeGroup', 'Income', 'IncomeBracket', 'CTR', 'Conversion Rate', 'Engagement Rate', 'High Value User', 'Is Young User', 'Ad Engagement Category']].head(10))

In [ ]:
print("=== Encoding Categorical Variables ===\n")

from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()

categorical_cols = ['Gender', 'Location', 'Ad Type', 'Ad Topic']

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[f'{col}_Encoded'] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le
    print(f"  {col} encoded: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print("\nOne-Hot Encoding Preview:")
df_onehot = pd.get_dummies(df[categorical_cols], prefix=categorical_cols)
print(f"   Original columns: {len(categorical_cols)}")
print(f"   After One-Hot: {len(df_onehot.columns)} columns")
print(f"   New columns: {list(df_onehot.columns)}")

print("\nLabel Encoding is preferred here since tree-based models handle ordinal values well,")
print("and One-Hot Encoding would create too many sparse features for columns like 'Ad Topic'.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(df['Engagement Rate'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(df['Engagement Rate'].mean(), color='red', linestyle='--', label=f'Mean: {df["Engagement Rate"].mean():.3f}')
axes[0, 0].axvline(df['Engagement Rate'].median(), color='green', linestyle='--', label=f'Median: {df["Engagement Rate"].median():.3f}')
axes[0, 0].set_xlabel('Engagement Rate')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Engagement Rate Distribution')
axes[0, 0].legend()

high_value_by_location = df.groupby('Location')['High Value User'].mean() * 100
axes[0, 1].bar(high_value_by_location.index, high_value_by_location.values, color='coral')
axes[0, 1].set_xlabel('Location')
axes[0, 1].set_ylabel('% High Value Users')
axes[0, 1].set_title('High Value Users by Location')
for i, val in enumerate(high_value_by_location.values):
    axes[0, 1].text(i, val + 1, f'{val:.1f}%', ha='center')

engagement_cat_counts = df['Ad Engagement Category'].value_counts()
axes[1, 0].pie(engagement_cat_counts, labels=engagement_cat_counts.index, autopct='%1.1f%%',
               colors=['#e74c3c', '#f39c12', '#2ecc71', '#3498db'], startangle=90)
axes[1, 0].set_title('Engagement Category Distribution')

young_engagement = df.groupby('Is Young User')['Engagement Rate'].mean()
labels = ['Older (35+)', 'Young (<35)']
axes[1, 1].bar(labels, young_engagement.values, color=['#9b59b6', '#1abc9c'])
axes[1, 1].set_ylabel('Avg Engagement Rate')
axes[1, 1].set_title('Engagement Rate: Young vs Older Users')
for i, val in enumerate(young_engagement.values):
    axes[1, 1].text(i, val + 0.002, f'{val:.4f}', ha='center')

plt.tight_layout()
plt.suptitle('Feature Engineering Insights', fontsize=14, fontweight='bold', y=1.02)
plt.show()

In [ ]:
print("=== Final Dataset Summary ===\n")

print(f"Dataset Shape: {df.shape}")
print(f"\nAll Columns ({len(df.columns)}):")
print(df.columns.tolist())

print("\nNumerical Features Statistics:")
print(df[['Age', 'Income', 'CTR', 'Conversion Rate', 'Engagement Rate']].describe().round(4))

print("\nCategorical Features:")
for col in ['Gender', 'Location', 'Ad Type', 'Ad Topic', 'AgeGroup', 'IncomeBracket', 'Ad Engagement Category']:
    if col in df.columns:
        print(f"   {col}: {df[col].nunique()} unique values")

print("\nDataset is ready for machine learning!")

## e. Outlier Handling (Numeric Features Only)
Detect outliers using boxplots, compare multiple approaches (IQR, Z-Score, Winsorization), and select the best one for model training.

In [ ]:
numeric_cols_for_outlier = ['Age', 'Income', 'CTR', 'Conversion Rate']

fig, axes = plt.subplots(1, len(numeric_cols_for_outlier), figsize=(5 * len(numeric_cols_for_outlier), 5))

for i, col in enumerate(numeric_cols_for_outlier):
    bp = axes[i].boxplot(df[col].dropna(), patch_artist=True,
                         boxprops=dict(facecolor='lightblue', color='navy'),
                         medianprops=dict(color='red', linewidth=2),
                         whiskerprops=dict(color='navy'),
                         flierprops=dict(marker='o', color='red', markersize=5))
    axes[i].set_title(f'Boxplot of {col}')
    axes[i].set_ylabel(col)

    # Count outliers using IQR
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_count = ((df[col] < lower) | (df[col] > upper)).sum()
    axes[i].set_xlabel(f'Outliers: {outlier_count}')

plt.tight_layout()
plt.suptitle('Boxplots for Outlier Detection', fontsize=14, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# Removes rows where values fall outside 1.5*IQR from Q1/Q3
df_iqr = df.copy()
print("=== Approach 1: IQR Method ===\n")
print(f"Original dataset size: {len(df_iqr)}")

for col in numeric_cols_for_outlier:
    Q1 = df_iqr[col].quantile(0.25)
    Q3 = df_iqr[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    before = len(df_iqr)
    df_iqr = df_iqr[(df_iqr[col] >= lower_bound) & (df_iqr[col] <= upper_bound)]
    removed = before - len(df_iqr)

    print(f"  {col}: bounds [{lower_bound:.2f}, {upper_bound:.2f}] -> removed {removed} rows")

print(f"\nDataset size after IQR removal: {len(df_iqr)}")
print(f"Total rows removed: {len(df) - len(df_iqr)} ({(len(df) - len(df_iqr)) / len(df) * 100:.1f}%)")
print(f"\nDescriptive stats after IQR method:")
print(df_iqr[numeric_cols_for_outlier].describe().round(4))

In [ ]:
# Removes rows where Z-score exceeds threshold (|z| > 3)
from scipy import stats

df_zscore = df.copy()
print("=== Approach 2: Z-Score Method (threshold = 3) ===\n")
print(f"Original dataset size: {len(df_zscore)}")

for col in numeric_cols_for_outlier:
    z_scores = np.abs(stats.zscore(df_zscore[col].dropna()))
    before = len(df_zscore)
    df_zscore = df_zscore[np.abs(stats.zscore(df_zscore[col])) <= 3]
    removed = before - len(df_zscore)
    print(f"  {col}: removed {removed} rows with |z| > 3")

print(f"\nDataset size after Z-Score removal: {len(df_zscore)}")
print(f"Total rows removed: {len(df) - len(df_zscore)} ({(len(df) - len(df_zscore)) / len(df) * 100:.1f}%)")
print(f"\nDescriptive stats after Z-Score method:")
print(df_zscore[numeric_cols_for_outlier].describe().round(4))

In [ ]:
# Caps extreme values at specified percentiles instead of removing rows
from scipy.stats import mstats

df_winsor = df.copy()
print("=== Approach 3: Winsorization (Capping at 5th and 95th percentile) ===\n")
print(f"Dataset size remains: {len(df_winsor)} (no rows removed)\n")

for col in numeric_cols_for_outlier:
    lower_cap = df_winsor[col].quantile(0.05)
    upper_cap = df_winsor[col].quantile(0.95)

    outliers_below = (df_winsor[col] < lower_cap).sum()
    outliers_above = (df_winsor[col] > upper_cap).sum()

    df_winsor[col] = df_winsor[col].clip(lower=lower_cap, upper=upper_cap)
    print(f"  {col}: capped to [{lower_cap:.2f}, {upper_cap:.2f}] (affected: {outliers_below + outliers_above} values)")

print(f"\nDescriptive stats after Winsorization:")
print(df_winsor[numeric_cols_for_outlier].describe().round(4))

In [ ]:
print("=== Comparison of Outlier Handling Approaches ===\n")

comparison_data = {
    'Approach': ['Original', 'IQR Method', 'Z-Score Method', 'Winsorization'],
    'Rows Remaining': [len(df), len(df_iqr), len(df_zscore), len(df_winsor)],
    'Data Loss (%)': [
        0,
        round((len(df) - len(df_iqr)) / len(df) * 100, 2),
        round((len(df) - len(df_zscore)) / len(df) * 100, 2),
        0  # Winsorization retains all rows
    ]
}

for approach_name, approach_df in [('Original', df), ('IQR', df_iqr), ('Z-Score', df_zscore), ('Winsorization', df_winsor)]:
    comparison_data[f'Income Mean'] = comparison_data.get(f'Income Mean', [])
    comparison_data[f'Income Mean'].append(round(approach_df['Income'].mean(), 2))
    comparison_data[f'Income Std'] = comparison_data.get(f'Income Std', [])
    comparison_data[f'Income Std'].append(round(approach_df['Income'].std(), 2))

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
titles = ['Original', 'After IQR', 'After Z-Score', 'After Winsorization']
datasets = [df, df_iqr, df_zscore, df_winsor]

for i, (title, data) in enumerate(zip(titles, datasets)):
    data[numeric_cols_for_outlier].boxplot(ax=axes[i])
    axes[i].set_title(title)
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.suptitle('Outlier Handling - Approach Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.show()

print("\nObservations:")
print(f"  - IQR Method: Removed {len(df) - len(df_iqr)} rows ({(len(df) - len(df_iqr))/len(df)*100:.1f}% data loss)")
print(f"  - Z-Score Method: Removed {len(df) - len(df_zscore)} rows ({(len(df) - len(df_zscore))/len(df)*100:.1f}% data loss)")
print(f"  - Winsorization: Retained all rows, capped extreme values")

In [ ]:
# IQR is selected because:
# - It is robust and does not assume any particular distribution
# - Income is right-skewed, making Z-Score less appropriate
# - Winsorization distorts original values which can introduce bias
# - IQR targets only genuine extreme observations

print("=== Selected Approach: IQR Method ===\n")
print("Justification:")
print("  1. IQR is robust and does not assume any particular distribution")
print("  2. Income is right-skewed, making Z-Score less appropriate")
print("  3. Winsorization alters actual values which can introduce bias")
print("  4. IQR-based removal targets only extreme observations\n")

df_clean = df.copy()
rows_before = len(df_clean)

for col in numeric_cols_for_outlier:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]

print(f"Rows before: {rows_before}")
print(f"Rows after:  {len(df_clean)}")
print(f"Rows removed: {rows_before - len(df_clean)}")

df = df_clean.copy()
df.reset_index(drop=True, inplace=True)
print(f"\nDataframe updated and index reset. Shape: {df.shape}")

In [ ]:
print("=== Descriptive Statistics: Before vs After Outlier Handling ===\n")

after_stats = df[numeric_cols_for_outlier].describe().round(4)

print("BEFORE Outlier Handling:")
print(baseline_stats[numeric_cols_for_outlier])
print("\nAFTER Outlier Handling (IQR):")
print(after_stats)

fig, axes = plt.subplots(2, len(numeric_cols_for_outlier), figsize=(5 * len(numeric_cols_for_outlier), 8))

for i, col in enumerate(numeric_cols_for_outlier):
    # Before (top row) — df_iqr used as original reference since df is now cleaned
    axes[0, i].boxplot(df_iqr[col].dropna() if col in df_iqr.columns else [], patch_artist=True,
                       boxprops=dict(facecolor='#ffcccc'))
    axes[0, i].set_title(f'{col}\n(Before)')
    axes[0, i].set_ylabel(col)

    axes[1, i].boxplot(df[col].dropna(), patch_artist=True,
                       boxprops=dict(facecolor='#ccffcc'))
    axes[1, i].set_title(f'{col}\n(After IQR)')
    axes[1, i].set_ylabel(col)

plt.tight_layout()
plt.suptitle('Before vs After Outlier Handling', fontsize=14, fontweight='bold', y=1.02)
plt.show()

print("\nOutlier handling complete. The cleaned dataset is ready for model training.")

## H-K: Data Preprocessing Pipeline

Complete preprocessing: feature engineering, reduction, normality testing, and scaling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from scipy import stats
import pickle
import warnings
warnings.filterwarnings('ignore')

df_model = pd.read_csv('social_media_ads_with_target.csv')
print('Dataset loaded successfully')
print(f'Shape: {df_model.shape}')
print(f'Columns: {df_model.columns.tolist()}')
print(f'\nTarget distribution:')
print(df_model['Purchased'].value_counts())

In [ ]:
print('=== FEATURE PREPARATION ===')

# Separate categorical and numeric features
categorical_features = df_model.select_dtypes(include=['object']).columns.tolist()
numeric_features = df_model.select_dtypes(include=[np.number]).columns.tolist()
numeric_features = [col for col in numeric_features if col != 'Purchased']

print(f'\nCategorical features: {categorical_features}')
print(f'Numeric features: {numeric_features}')

# Encode categorical variables
df_encoded = df_model.copy()
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

print(f'\nDataset shape after encoding: {df_encoded.shape}')
print(df_encoded.head())

In [ ]:
print('\n=== H. CORRELATION ANALYSIS & FEATURE REDUCTION ===')

all_numeric = categorical_features + numeric_features  # All encoded features
corr_matrix = df_encoded[all_numeric + numeric_features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix of All Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Find highly correlated features (excluding target)
feature_list = all_numeric + numeric_features
print('\nHighly correlated feature pairs (|r| >= 0.95):')
highly_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) >= 0.95:
            col1 = corr_matrix.columns[i]
            col2 = corr_matrix.columns[j]
            corr_val = corr_matrix.iloc[i, j]
            print(f'  {col1} <-> {col2}: {corr_val:.4f}')
            highly_corr_pairs.append((col1, col2, corr_val))

if not highly_corr_pairs:
    print('  No highly correlated pairs found.')

# Features to keep
cols_to_drop = set()
for col1, col2, _ in highly_corr_pairs:
    cols_to_drop.add(col2)

df_reduced = df_encoded.drop(columns=list(cols_to_drop)) if cols_to_drop else df_encoded.copy()
print(f'\nColumns dropped: {list(cols_to_drop)}')
print(f'Features after correlation filtering: {len(df_reduced.columns) - 1}')  # -1 for target

In [ ]:
print('\n=== PCA ANALYSIS FOR DIMENSIONALITY REDUCTION ===')

feature_cols_pca = [col for col in df_reduced.columns if col != 'Purchased']
X_for_pca = df_reduced[feature_cols_pca].copy()

# Standardize for PCA
scaler_pca = StandardScaler()
X_scaled_pca = scaler_pca.fit_transform(X_for_pca)

# Apply PCA
pca = PCA()
pca.fit(X_scaled_pca)

# Analyze variance
cumsum_var = np.cumsum(pca.explained_variance_ratio_)
n_components_95 = np.argmax(cumsum_var >= 0.95) + 1

print(f'\nTotal features: {len(feature_cols_pca)}')
print(f'Components for 95% variance: {n_components_95}')
print(f'\nExplained variance (first 10):')  
for i, var in enumerate(pca.explained_variance_ratio_[:10], 1):
    print(f'  PC{i}: {var:.4f} ({cumsum_var[i-1]:.4f} cumulative)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, min(11, len(feature_cols_pca)+1)), 
            pca.explained_variance_ratio_[:10])
axes[0].set_title('PCA Explained Variance')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Ratio')

axes[1].plot(range(1, min(11, len(feature_cols_pca)+1)), 
             cumsum_var[:10], 'bo-')
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance')
axes[1].legend()

plt.tight_layout()
plt.show()

print('\nNote: Original features retained for interpretability.')

In [ ]:
print('\n=== I. NORMALITY TEST: SKEWNESS ANALYSIS (BEFORE SCALING) ===')

# Select numeric features only for skewness analysis
feature_cols = [col for col in df_reduced.columns if col != 'Purchased']
numeric_feature_cols = [col for col in feature_cols if df_reduced[col].dtype in [np.int64, np.float64]]

skewness_data = {}
for col in numeric_feature_cols:
    skew = df_reduced[col].skew()
    kurt = df_reduced[col].kurtosis()
    skewness_data[col] = {'Skewness': skew, 'Kurtosis': kurt}

skew_df = pd.DataFrame(skewness_data).T.round(4)
print('\nSkewness and Kurtosis:')
print(skew_df)

# Identify highly skewed features
highly_skewed = [col for col in numeric_feature_cols if abs(df_reduced[col].skew()) > 1]
print(f'\nHighly skewed features (|skew| > 1): {highly_skewed}')

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(numeric_feature_cols):
    if idx < len(axes):
        axes[idx].hist(df_reduced[col], bins=30, edgecolor='black', alpha=0.7)
        skew_val = df_reduced[col].skew()
        axes[idx].set_title(f'{col} (Skew: {skew_val:.3f})')
        axes[idx].set_ylabel('Frequency')
        axes[idx].grid(alpha=0.3)

for idx in range(len(numeric_feature_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.suptitle('Feature Distributions BEFORE Scaling', fontsize=14, fontweight='bold', y=1.00)
plt.show()

In [ ]:
print('\n=== LOG TRANSFORMATION FOR HIGHLY SKEWED FEATURES ===')

df_transformed = df_reduced.copy()

# Apply log1p to highly skewed features
for col in highly_skewed:
    if (df_transformed[col] >= 0).all():
        before_skew = df_transformed[col].skew()
        df_transformed[col] = np.log1p(df_transformed[col])
        after_skew = df_transformed[col].skew()
        print(f'{col}: {before_skew:.4f} -> {after_skew:.4f}')

if not highly_skewed:
    print('No highly skewed features found.')

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(numeric_feature_cols):
    if idx < len(axes):
        axes[idx].hist(df_transformed[col], bins=30, edgecolor='black', alpha=0.7, color='green')
        skew_val = df_transformed[col].skew()
        axes[idx].set_title(f'{col} (Skew: {skew_val:.3f})')
        axes[idx].set_ylabel('Frequency')
        axes[idx].grid(alpha=0.3)

for idx in range(len(numeric_feature_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.suptitle('Feature Distributions AFTER Log Transformation', fontsize=14, fontweight='bold', y=1.00)
plt.show()

In [ ]:
print('\n=== J. SCALING AND NORMALIZATION WITH WEIGHT SAVING ===')

feature_cols_final = [col for col in df_transformed.columns if col != 'Purchased']
X = df_transformed[feature_cols_final].copy()
y = df_transformed['Purchased'].copy()

print(f'\nFeatures for modeling: {feature_cols_final}')
print(f'Feature statistics before scaling:')
print(X.describe().round(4))

scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(X)

# Convert to DataFrame
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols_final)

print(f'\nFeature statistics after MinMaxScaler (0-1 range):')
print(X_scaled_df.describe().round(4))

with open('scaler_weights.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print(f'\nScaler saved to: scaler_weights.pkl')
print(f'Min: {scaler.data_min_}')
print(f'Max: {scaler.data_max_}')

In [ ]:
print('\n=== K. NORMALITY TEST: SKEWNESS ANALYSIS (AFTER SCALING) ===')

skew_after = {}
for col in numeric_feature_cols:
    skew_after[col] = X_scaled_df[col].skew()

print('\nSkewness after scaling (MinMaxScaler preserves skewness):')
for col in numeric_feature_cols:
    skew_before = skewness_data[col]['Skewness']
    skew_now = skew_after[col]
    print(f'{col:20} Before: {skew_before:8.4f}  After: {skew_now:8.4f}')

In [ ]:
from sklearn.model_selection import train_test_split

print('=== TRAIN-TEST SPLIT ===')

# Split data (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Total samples: {len(X_scaled_df)}')
print(f'Training set: {len(X_train)} ({len(X_train)/len(X_scaled_df)*100:.1f}%)')
print(f'Testing set: {len(X_test)} ({len(X_test)/len(X_scaled_df)*100:.1f}%)')
print('Class distribution in training set:')
print(y_train.value_counts())
print('Class distribution in testing set:')
print(y_test.value_counts())
print('Data prepared for model training!')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import pickle

print('=' * 80)
print('MODEL 1: LOGISTIC REGRESSION')
print('=' * 80)

# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_model.fit(X_train, y_train)

# Predictions
y_train_pred_lr = lr_model.predict(X_train)
y_test_pred_lr = lr_model.predict(X_test)
y_test_pred_proba_lr = lr_model.predict_proba(X_test)[:, 1]

# Evaluation
train_acc_lr = accuracy_score(y_train, y_train_pred_lr)
test_acc_lr = accuracy_score(y_test, y_test_pred_lr)
precision_lr = precision_score(y_test, y_test_pred_lr, zero_division=0)
recall_lr = recall_score(y_test, y_test_pred_lr, zero_division=0)
f1_lr = f1_score(y_test, y_test_pred_lr, zero_division=0)
auc_lr = roc_auc_score(y_test, y_test_pred_proba_lr)

print(f'\nTraining Accuracy: {train_acc_lr:.4f}')
print(f'Testing Accuracy:  {test_acc_lr:.4f}')
print(f'Precision: {precision_lr:.4f}')
print(f'Recall:    {recall_lr:.4f}')
print(f'F1-Score:  {f1_lr:.4f}')
print(f'ROC-AUC:   {auc_lr:.4f}')

print(f'\nConfusion Matrix:')
cm_lr = confusion_matrix(y_test, y_test_pred_lr)
print(cm_lr)

print(f'\nClassification Report:')
print(classification_report(y_test, y_test_pred_lr))

lr_model_path = 'logistic_regression_model.pkl'
with open(lr_model_path, 'wb') as f:
    pickle.dump(lr_model, f)
print(f'\nModel saved to: {lr_model_path}')

In [ ]:
from sklearn.ensemble import RandomForestClassifier

print('\n' + '=' * 80)
print('MODEL 2: RANDOM FOREST CLASSIFIER')
print('=' * 80)

# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predictions
y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)
y_test_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Evaluation
train_acc_rf = accuracy_score(y_train, y_train_pred_rf)
test_acc_rf = accuracy_score(y_test, y_test_pred_rf)
precision_rf = precision_score(y_test, y_test_pred_rf, zero_division=0)
recall_rf = recall_score(y_test, y_test_pred_rf, zero_division=0)
f1_rf = f1_score(y_test, y_test_pred_rf, zero_division=0)
auc_rf = roc_auc_score(y_test, y_test_pred_proba_rf)

print(f'\nTraining Accuracy: {train_acc_rf:.4f}')
print(f'Testing Accuracy:  {test_acc_rf:.4f}')
print(f'Precision: {precision_rf:.4f}')
print(f'Recall:    {recall_rf:.4f}')
print(f'F1-Score:  {f1_rf:.4f}')
print(f'ROC-AUC:   {auc_rf:.4f}')

print(f'\nConfusion Matrix:')
cm_rf = confusion_matrix(y_test, y_test_pred_rf)
print(cm_rf)

print(f'\nClassification Report:')
print(classification_report(y_test, y_test_pred_rf))

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(f'\nTop 10 Important Features:')
print(feature_importance.head(10))

rf_model_path = 'random_forest_model.pkl'
with open(rf_model_path, 'wb') as f:
    pickle.dump(rf_model, f)
print(f'\nModel saved to: {rf_model_path}')

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

print('\n' + '=' * 80)
print('MODEL 3: GRADIENT BOOSTING CLASSIFIER')
print('=' * 80)

# Train Gradient Boosting
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
gb_model.fit(X_train, y_train)

# Predictions
y_train_pred_gb = gb_model.predict(X_train)
y_test_pred_gb = gb_model.predict(X_test)
y_test_pred_proba_gb = gb_model.predict_proba(X_test)[:, 1]

# Evaluation
train_acc_gb = accuracy_score(y_train, y_train_pred_gb)
test_acc_gb = accuracy_score(y_test, y_test_pred_gb)
precision_gb = precision_score(y_test, y_test_pred_gb, zero_division=0)
recall_gb = recall_score(y_test, y_test_pred_gb, zero_division=0)
f1_gb = f1_score(y_test, y_test_pred_gb, zero_division=0)
auc_gb = roc_auc_score(y_test, y_test_pred_proba_gb)

print(f'\nTraining Accuracy: {train_acc_gb:.4f}')
print(f'Testing Accuracy:  {test_acc_gb:.4f}')
print(f'Precision: {precision_gb:.4f}')
print(f'Recall:    {recall_gb:.4f}')
print(f'F1-Score:  {f1_gb:.4f}')
print(f'ROC-AUC:   {auc_gb:.4f}')

print(f'\nConfusion Matrix:')
cm_gb = confusion_matrix(y_test, y_test_pred_gb)
print(cm_gb)

print(f'\nClassification Report:')
print(classification_report(y_test, y_test_pred_gb))

# Feature importance
feature_importance_gb = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': gb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(f'\nTop 10 Important Features:')
print(feature_importance_gb.head(10))

gb_model_path = 'gradient_boosting_model.pkl'
with open(gb_model_path, 'wb') as f:
    pickle.dump(gb_model, f)
print(f'\nModel saved to: {gb_model_path}')

In [ ]:
from sklearn.svm import SVC

print('\n' + '=' * 80)
print('MODEL 4: SUPPORT VECTOR MACHINE (SVM)')
print('=' * 80)

# Train SVM
svm_model = SVC(kernel='rbf', probability=True, random_state=42, gamma='scale')
svm_model.fit(X_train, y_train)

# Predictions
y_train_pred_svm = svm_model.predict(X_train)
y_test_pred_svm = svm_model.predict(X_test)
y_test_pred_proba_svm = svm_model.predict_proba(X_test)[:, 1]

# Evaluation
train_acc_svm = accuracy_score(y_train, y_train_pred_svm)
test_acc_svm = accuracy_score(y_test, y_test_pred_svm)
precision_svm = precision_score(y_test, y_test_pred_svm, zero_division=0)
recall_svm = recall_score(y_test, y_test_pred_svm, zero_division=0)
f1_svm = f1_score(y_test, y_test_pred_svm, zero_division=0)
auc_svm = roc_auc_score(y_test, y_test_pred_proba_svm)

print(f'\nTraining Accuracy: {train_acc_svm:.4f}')
print(f'Testing Accuracy:  {test_acc_svm:.4f}')
print(f'Precision: {precision_svm:.4f}')
print(f'Recall:    {recall_svm:.4f}')
print(f'F1-Score:  {f1_svm:.4f}')
print(f'ROC-AUC:   {auc_svm:.4f}')

print(f'\nConfusion Matrix:')
cm_svm = confusion_matrix(y_test, y_test_pred_svm)
print(cm_svm)

print(f'\nClassification Report:')
print(classification_report(y_test, y_test_pred_svm))

svm_model_path = 'svm_model.pkl'
with open(svm_model_path, 'wb') as f:
    pickle.dump(svm_model, f)
print(f'\nModel saved to: {svm_model_path}')

In [ ]:
try:
    import xgboost as xgb
    
    print('\n' + '=' * 80)
    print('MODEL 5: XGBOOST CLASSIFIER')
    print('=' * 80)
    
    # Train XGBoost
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    xgb_model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred_xgb = xgb_model.predict(X_train)
    y_test_pred_xgb = xgb_model.predict(X_test)
    y_test_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
    
    # Evaluation
    train_acc_xgb = accuracy_score(y_train, y_train_pred_xgb)
    test_acc_xgb = accuracy_score(y_test, y_test_pred_xgb)
    precision_xgb = precision_score(y_test, y_test_pred_xgb, zero_division=0)
    recall_xgb = recall_score(y_test, y_test_pred_xgb, zero_division=0)
    f1_xgb = f1_score(y_test, y_test_pred_xgb, zero_division=0)
    auc_xgb = roc_auc_score(y_test, y_test_pred_proba_xgb)
    
    print(f'\nTraining Accuracy: {train_acc_xgb:.4f}')
    print(f'Testing Accuracy:  {test_acc_xgb:.4f}')
    print(f'Precision: {precision_xgb:.4f}')
    print(f'Recall:    {recall_xgb:.4f}')
    print(f'F1-Score:  {f1_xgb:.4f}')
    print(f'ROC-AUC:   {auc_xgb:.4f}')
    
    print(f'\nConfusion Matrix:')
    cm_xgb = confusion_matrix(y_test, y_test_pred_xgb)
    print(cm_xgb)
    
    print(f'\nClassification Report:')
    print(classification_report(y_test, y_test_pred_xgb))
    
    # Feature importance
    feature_importance_xgb = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': xgb_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print(f'\nTop 10 Important Features:')
    print(feature_importance_xgb.head(10))
    
    xgb_model_path = 'xgboost_model.pkl'
    with open(xgb_model_path, 'wb') as f:
        pickle.dump(xgb_model, f)
    print(f'\nModel saved to: {xgb_model_path}')
    
except ImportError:
    print('XGBoost not installed. Skipping this model.')
    train_acc_xgb = test_acc_xgb = 0

## M. Model Evaluation & Comparison

In [ ]:
print('\n' + '=' * 80)
print('MODEL PERFORMANCE COMPARISON')
print('=' * 80)

# Compile results
results = {
    'Model': ['Logistic Regression', 'Random Forest', 'Gradient Boosting', 'SVM', 'XGBoost'],
    'Train Accuracy': [train_acc_lr, train_acc_rf, train_acc_gb, train_acc_svm, train_acc_xgb],
    'Test Accuracy': [test_acc_lr, test_acc_rf, test_acc_gb, test_acc_svm, test_acc_xgb],
    'Precision': [precision_lr, precision_rf, precision_gb, precision_svm, precision_xgb],
    'Recall': [recall_lr, recall_rf, recall_gb, recall_svm, recall_xgb],
    'F1-Score': [f1_lr, f1_rf, f1_gb, f1_svm, f1_xgb],
    'ROC-AUC': [auc_lr, auc_rf, auc_gb, auc_svm, auc_xgb]
}

results_df = pd.DataFrame(results).round(4)
print('\n')
print(results_df.to_string(index=False))

# Find best model
best_idx = results_df['Test Accuracy'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
best_accuracy = results_df.loc[best_idx, 'Test Accuracy']

print(f'\n' + '=' * 80)
print(f'BEST MODEL: {best_model_name}')
print(f'Test Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)')
print('=' * 80)

if best_accuracy >= 0.90:
    print(f'\nSUCCESS! Model accuracy is >= 90% requirement!')
else:
    print(f'\nNote: Model accuracy is {best_accuracy*100:.2f}%, below 90% target.')
    print('Consider hyperparameter tuning or ensemble methods.')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
axes[0].barh(results_df['Model'], results_df['Test Accuracy'], color='steelblue')
axes[0].axvline(x=0.90, color='red', linestyle='--', label='90% Target')
axes[0].set_xlabel('Test Accuracy')
axes[0].set_title('Model Test Accuracy Comparison')
axes[0].legend()
axes[0].set_xlim([0.7, 1.0])
axes[0].grid(axis='x', alpha=0.3)

# Precision-Recall-F1 comparison
x = np.arange(len(results_df))
width = 0.25
axes[1].bar(x - width, results_df['Precision'], width, label='Precision')
axes[1].bar(x, results_df['Recall'], width, label='Recall')
axes[1].bar(x + width, results_df['F1-Score'], width, label='F1-Score')
axes[1].set_ylabel('Score')
axes[1].set_title('Precision, Recall, F1-Score Comparison')
axes[1].set_xticks(x)
axes[1].set_xticklabels(results_df['Model'], rotation=45, ha='right')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

results_df.to_csv('model_comparison_results.csv', index=False)
print('\nModel comparison results saved to: model_comparison_results.csv')

In [ ]:
print('\n' + '=' * 80)
print('SAVING BEST MODEL AND ASSOCIATED OBJECTS')
print('=' * 80)

# Select best model based on test accuracy
models_dict = {
    'Logistic Regression': lr_model,
    'Random Forest': rf_model,
    'Gradient Boosting': gb_model,
    'SVM': svm_model,
}

# Add XGBoost if it was trained (accuracy > 0)
if train_acc_xgb > 0:
    models_dict['XGBoost'] = xgb_model

best_model = models_dict[best_model_name]

best_model_path = 'best_model.pkl'
with open(best_model_path, 'wb') as f:
    pickle.dump(best_model, f)
print(f'Best model ({best_model_name}) saved to: {best_model_path}')

feature_names_path = 'feature_names.pkl'
with open(feature_names_path, 'wb') as f:
    pickle.dump(feature_cols, f)
print(f'Feature names saved to: {feature_names_path}')

model_info = {
    'best_model_name': best_model_name,
    'best_test_accuracy': float(best_accuracy),
    'best_train_accuracy': float(results_df.loc[best_idx, 'Train Accuracy']),
    'best_precision': float(results_df.loc[best_idx, 'Precision']),
    'best_recall': float(results_df.loc[best_idx, 'Recall']),
    'best_f1_score': float(results_df.loc[best_idx, 'F1-Score']),
    'best_roc_auc': float(results_df.loc[best_idx, 'ROC-AUC']),
    'feature_cols': feature_cols,
    'scaler_path': 'scaler_weights.pkl',
    'model_path': best_model_path,
    'feature_names_path': feature_names_path,
    'all_models': {
        'logistic_regression': 'logistic_regression_model.pkl',
        'random_forest': 'random_forest_model.pkl',
        'gradient_boosting': 'gradient_boosting_model.pkl',
        'svm': 'svm_model.pkl',
        'xgboost': 'xgboost_model.pkl' if train_acc_xgb > 0 else None
    }
}

import json
model_info_path = 'model_info.json'
with open(model_info_path, 'w') as f:
    json.dump(model_info, f, indent=2)
print(f'Model information saved to: {model_info_path}')

print('\n' + '=' * 80)
print('PROJECT COMPLETE!')
print('=' * 80)
print('\nFiles created for demo purposes:')
print('  1. best_model.pkl - Best trained model')
print('  2. scaler_weights.pkl - Feature scaler for preprocessing')
print('  3. feature_names.pkl - List of feature column names')
print('  4. model_info.json - Model metadata and paths')
print('  5. model_comparison_results.csv - All model performance metrics')
print('  6. Individual model files for each algorithm')
print('\nTo use the model for predictions:')
print('  1. Load the scaler: pickle.load(open("scaler_weights.pkl", "rb"))')
print('  2. Scale new data: X_new_scaled = scaler.transform(X_new)')
print('  3. Load the model: pickle.load(open("best_model.pkl", "rb"))')
print('  4. Make predictions: predictions = model.predict(X_new_scaled)')